# Notebook 1 — Evaluation, Ablation, Bootstrap CIsReproduces the single-backbone numbers stored in the checkpoints, then evaluatesall 7 equal-weight softmax-average configurations of{E = EfficientNet-B1, C = ConvNeXt-Tiny, S = Swin-Tiny}.**What this notebook establishes (and the paper must say):** the reported system is*not* a jointly trained fusion network. It is three independently fine-tunedbackbones whose softmax outputs are averaged with equal weights. No fusion layer,projection or fusion-MLP exists in any checkpoint.Every number printed here is also written to disk. No result exists only as a plot.**Test set — read this before quoting any number.** Notebook 2 found that**162 of the 727 test images (22.3%) have a perceptual-hash near-duplicate(pHash, Hamming d<=5) in train or val.** Accuracy on the full 727 thereforeoverstates generalisation. Every reported metric in this notebook is computed onthe **de-duplicated 565-image subset** (`test_clean_manifest.csv`), which is thepaper's primary test set. Two deliberate exceptions:* **Section 6** (the reproduction gate) runs on the full 727, because the  checkpoints stored their accuracy on that basis and only that set is comparable.* **Section 15** recomputes the full-test numbers as a contamination appendix, to  quantify how much the leakage inflated things. Those are not results.Set `CLEAN_MANIFEST = None` in section 1 to evaluate on the full set instead; thenotebook then prints a warning on every run.**Outputs** (to `/kaggle/working/`): `cache_*.npz`, `results.json`,`confusion_*.csv`, `fig3_model_comparison.pdf`, `fig4_roc.pdf`,`fig5_confusion_matrix.pdf`, `per_class_metrics.csv`, `ablation_table.csv`,`results_full_contaminated.json`, `contamination_comparison.csv`.

## 1. Setup — paths, seed, determinism

In [ ]:
# ============================ EDIT THESE ============================DATA_ROOT = '/kaggle/input/datasets/jainmanav20/ipd-skin-dataset/IPD_Datasets_Final_Split'CKPT_ROOT = '/kaggle/input/datasets/jainmanav20/checkpoints'OUT_DIR   = '/kaggle/working'# De-duplicated test manifest written by Notebook 2 (test_clean_manifest.csv).# Notebook 2 found 162/727 test images with a near-duplicate (pHash d<=5) in# train/val, so the 565-image clean subset is the PRIMARY reported test set.# Upload Notebook 2's output as a Kaggle dataset and point this at it, or set# CLEAN_MANIFEST = None to fall back to the full 727-image test set.CLEAN_MANIFEST = '/kaggle/input/datasets/jainmanav20/skinsure-dedup/test_clean_manifest.csv'# ====================================================================# Expected layout:#   DATA_ROOT/{train,val,test}/<ClassName>/*.jpg#   CKPT_ROOT/efficientnet_b1_5cls_skin.pth#   CKPT_ROOT/convnext_tiny_5cls_skin.pth#   CKPT_ROOT/swin_tiny_5cls_skin.pth# If your checkpoint dataset keeps the nested folders, set e.g.#   CKPT_EFF = CKPT_ROOT + '/efficientNet/efficientnet_b1_5cls_skin.pth'# in the next cell instead.import os, sys, json, time, random, warningsfrom pathlib import Pathfrom collections import Counter, OrderedDictimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport seaborn as snsimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import DataLoaderfrom torchvision import transformsfrom torchvision.datasets import ImageFolderimport timmfrom sklearn.metrics import (    accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef,    roc_auc_score, roc_curve, auc, confusion_matrix, precision_recall_fscore_support,)from sklearn.preprocessing import label_binarizewarnings.filterwarnings('ignore')SEED = 42random.seed(SEED); np.random.seed(SEED)torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)# Deterministic evaluation (the original training runs disabled this for speed).torch.backends.cudnn.deterministic = Truetorch.backends.cudnn.benchmark     = FalseDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'print(f'Device : {DEVICE}')print(f'GPU    : {GPU_NAME}')print(f'torch  : {torch.__version__} | timm : {timm.__version__}')DATA_ROOT = Path(DATA_ROOT); CKPT_ROOT = Path(CKPT_ROOT); OUT = Path(OUT_DIR)OUT.mkdir(parents=True, exist_ok=True)

## 2. Checkpoints and the 5 paper classesClass order is taken from the checkpoints (`ckpt['classes']`), **not** from thefolder listing, so indices always match the trained heads. The corpus on disk alsocontains Acne, Lichen_Planus, Psoriasis and Vitiligo; those folders are ignored.

In [ ]:
def _resolve(*cands):    for c in cands:        p = Path(c)        if p.exists():            return p    raise FileNotFoundError(f'none of these exist: {cands}')CKPT_EFF  = _resolve(CKPT_ROOT/'efficientnet_b1_5cls_skin.pth',                     CKPT_ROOT/'efficientNet'/'efficientnet_b1_5cls_skin.pth')CKPT_CNXT = _resolve(CKPT_ROOT/'convnext_tiny_5cls_skin.pth',                     CKPT_ROOT/'convNext'/'convnext_tiny_5cls_skin.pth')CKPT_SWIN = _resolve(CKPT_ROOT/'swin_tiny_5cls_skin.pth',                     CKPT_ROOT/'swin'/'swin_tiny_5cls_skin.pth')for p in (CKPT_EFF, CKPT_CNXT, CKPT_SWIN):    print('found:', p)_ck = torch.load(CKPT_EFF, map_location='cpu', weights_only=False)CLASSES = list(_ck['classes'])          # canonical order from the checkpointNUM_CLS = len(CLASSES)PAPER_CLASSES = set(CLASSES)print(f'\nCLASSES ({NUM_CLS}) from checkpoint: {CLASSES}')assert PAPER_CLASSES == {'Basal_Cell_Carcinoma','Eczema','Keratosis','Melanoma','Tinea'}# All three checkpoints must agree on class order, else index mixing would silently corrupt results.for nm, p in [('convnext', CKPT_CNXT), ('swin', CKPT_SWIN)]:    cls = list(torch.load(p, map_location='cpu', weights_only=False)['classes'])    assert cls == CLASSES, f'{nm} class order differs: {cls} vs {CLASSES}'print('All three checkpoints share the same class order.')MEL_IDX = CLASSES.index('Melanoma')BCC_IDX = CLASSES.index('Basal_Cell_Carcinoma')

## 3. Eval transform — copied verbatim from the training notebooksFrom `src/models/*_5class.ipynb` cell 4 (`val_tf`). Resize to a fixed224x224 (no centre crop), ToTensor, ImageNet normalisation. Matching this exactlyis what makes the reproduction check in section 6 meaningful.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD  = [0.229, 0.224, 0.225]# Verbatim from the training notebooks' val_tf (img_size = 224).val_tf = transforms.Compose([    transforms.Resize((224, 224)),    transforms.ToTensor(),    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),])class FiveClassImageFolder(ImageFolder):    # ImageFolder restricted to the 5 paper classes, with class_to_idx forced to the    # checkpoint's class order so target indices match the trained heads.    def find_classes(self, directory):        return list(CLASSES), {c: i for i, c in enumerate(CLASSES)}def make_loader(split, batch_size=32, workers=2):    ds = FiveClassImageFolder(DATA_ROOT/split, transform=val_tf)    assert ds.classes == CLASSES, f'{ds.classes} != {CLASSES}'    ld = DataLoader(ds, batch_size=batch_size, shuffle=False,                    num_workers=workers, pin_memory=True)    return ds, lddatasets, loaders = {}, {}for split in ['val', 'test']:    ds, ld = make_loader(split)    datasets[split], loaders[split] = ds, ld    cnt = Counter(ds.targets)    print(f'{split:>5}: n={len(ds):5d} | ' +          ' '.join(f'{c}={cnt[i]}' for i, c in enumerate(CLASSES)))SPLIT_COUNTS = {    s: {c: int(Counter(datasets[s].targets)[i]) for i, c in enumerate(CLASSES)}    for s in datasets}SPLIT_N = {s: len(datasets[s]) for s in datasets}

## 4. Model definitions — copied verbatim from the training notebooksEach is `timm` backbone (`num_classes=0`) + the same 6-layer head`LayerNorm -> Dropout -> Linear(feat,256) -> GELU -> Dropout(d/2) -> Linear(256,5)`.`dropout` / `drop_path_rate` are read from each checkpoint's own `config`, and allthree load with **`strict=True`**.Note what these definitions contain: `backbone.*` and `classifier.*` only. There isno projection to 512, no concatenation, no fusion LayerNorm and no fusion MLP.

In [ ]:
class EfficientSkinClassifier(nn.Module):    def __init__(self, num_classes, dropout=0.35, drop_path_rate=0.15, pretrained=False):        super().__init__()        self.backbone = timm.create_model(            'efficientnet_b1', pretrained=pretrained, num_classes=0,            drop_rate=dropout, drop_path_rate=drop_path_rate)        feat_dim = self.backbone.num_features            # 1280        self.classifier = nn.Sequential(            nn.LayerNorm(feat_dim), nn.Dropout(dropout),            nn.Linear(feat_dim, 256), nn.GELU(),            nn.Dropout(dropout/2), nn.Linear(256, num_classes))    def forward(self, x): return self.classifier(self.backbone(x))class ConvNeXtSkinClassifier(nn.Module):    def __init__(self, num_classes, dropout=0.4, drop_path_rate=0.2, pretrained=False):        super().__init__()        self.backbone = timm.create_model(            'convnext_tiny', pretrained=pretrained, num_classes=0,            drop_rate=dropout, drop_path_rate=drop_path_rate)        feat_dim = self.backbone.num_features            # 768        self.classifier = nn.Sequential(            nn.LayerNorm(feat_dim), nn.Dropout(dropout),            nn.Linear(feat_dim, 256), nn.GELU(),            nn.Dropout(dropout/2), nn.Linear(256, num_classes))    def forward(self, x): return self.classifier(self.backbone(x))class SwinSkinClassifier(nn.Module):    def __init__(self, model_name, num_classes, dropout=0.4, drop_path_rate=0.2, pretrained=False):        super().__init__()        self.backbone = timm.create_model(            model_name, pretrained=pretrained, num_classes=0,            drop_rate=dropout, drop_path_rate=drop_path_rate)        feat_dim = self.backbone.num_features            # 768        self.classifier = nn.Sequential(            nn.LayerNorm(feat_dim), nn.Dropout(dropout),            nn.Linear(feat_dim, 256), nn.GELU(),            nn.Dropout(dropout/2), nn.Linear(256, num_classes))    def forward(self, x): return self.classifier(self.backbone(x))def load_backbone(kind, path):    ckpt = torch.load(path, map_location='cpu', weights_only=False)    cfg  = ckpt['config']    kw = dict(num_classes=len(ckpt['classes']),              dropout=cfg['dropout'], drop_path_rate=cfg['drop_path_rate'])    if kind == 'eff':        model = EfficientSkinClassifier(**kw)    elif kind == 'cnxt':        model = ConvNeXtSkinClassifier(**kw)    elif kind == 'swin':        model = SwinSkinClassifier(model_name=cfg['model_name'], **kw)    else:        raise ValueError(kind)    missing, unexpected = model.load_state_dict(ckpt['model_state'], strict=True)    model = model.to(DEVICE, memory_format=torch.channels_last).eval()    # Sanity: the head must end in (5, 256) -- a per-backbone head, not a fused one.    assert tuple(model.classifier[5].weight.shape) == (len(ckpt['classes']), 256)    print(f"  loaded strict=True | {cfg['model_name']:32s} "          f"ckpt test_acc={ckpt['test_acc']*100:.2f}% macro_auc={ckpt['macro_auc']:.4f}")    return model, ckptprint('Loading checkpoints (strict=True):')eff_model,  CK_EFF  = load_backbone('eff',  CKPT_EFF)cnxt_model, CK_CNXT = load_backbone('cnxt', CKPT_CNXT)swin_model, CK_SWIN = load_backbone('swin', CKPT_SWIN)MODELS = OrderedDict([('E', eff_model), ('C', cnxt_model), ('S', swin_model)])MODEL_LONG = {'E': 'EfficientNet-B1', 'C': 'ConvNeXt-Tiny', 'S': 'Swin-Tiny'}CKPTS = {'E': CK_EFF, 'C': CK_CNXT, 'S': CK_SWIN}# Confirm no fusion/projection tensors exist anywhere (the paper's Fig. 2 claim).for k, ck in CKPTS.items():    keys = list(ck['model_state'].keys())    prefixes = sorted({x.split('.')[0] for x in keys})    fusion_like = [x for x in keys                   if not x.startswith('backbone')                   and any(t in x.lower() for t in ('fus', 'proj', 'mlp'))]    print(f'{MODEL_LONG[k]:16s} prefixes={prefixes} fusion/projection tensors={len(fusion_like)}')

## 5. Inference — cache softmax outputs, labels and file pathsOne forward pass per model per split. Everything downstream (ablation, CIs,tone analysis in Notebook 3) is computed from these caches, so nothing needs a GPUagain.

In [ ]:
@torch.no_grad()def infer(model, loader):    probs = []    for imgs, _ in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        probs.append(F.softmax(model(imgs), dim=1).float().cpu().numpy())    return np.concatenate(probs, axis=0)CACHE = {}   # CACHE[split][key] -> (n, 5) probsLABELS = {}PATHS  = {}for split in ['val', 'test']:    ds = datasets[split]    LABELS[split] = np.array(ds.targets, dtype=np.int64)    PATHS[split]  = np.array([s[0] for s in ds.samples], dtype=object)    CACHE[split]  = {}    for key, model in MODELS.items():        t0 = time.time()        p = infer(model, loaders[split])        CACHE[split][key] = p        assert p.shape == (len(ds), NUM_CLS), p.shape        # softmax rows must sum to 1        assert np.allclose(p.sum(1), 1.0, atol=1e-4), 'softmax rows do not sum to 1'        npz = OUT/f'cache_{MODEL_LONG[key].replace("-","_").lower()}_{split}.npz'        np.savez_compressed(npz, probs=p, labels=LABELS[split],                            paths=PATHS[split].astype('U'), classes=np.array(CLASSES))        print(f'{split:>5} {MODEL_LONG[key]:16s} {p.shape} {time.time()-t0:6.1f}s -> {npz.name}')# ---- de-duplication mask (primary test set) -------------------------------# Notebook 2 and this notebook may mount DATA_ROOT under different prefixes, so# match on '<ClassName>/<filename>' rather than the absolute path. A prefix# mismatch would otherwise silently keep every image and quietly report the# contaminated numbers.def _key(p):    p = Path(str(p))    return f'{p.parent.name}/{p.name}'TEST_KEYS = np.array([_key(p) for p in PATHS['test']], dtype=object)if CLEAN_MANIFEST is None:    CLEAN_MASK = np.ones(len(TEST_KEYS), dtype=bool)    EVAL_SET = 'full'    print('\nCLEAN_MANIFEST = None -> evaluating on the FULL test set.')    print('WARNING: Notebook 2 found 22.3% cross-split near-duplicates; '          'these numbers are contaminated.')else:    _mp = Path(CLEAN_MANIFEST)    if not _mp.exists():        raise FileNotFoundError(            f'CLEAN_MANIFEST not found: {_mp}\n'            'Run Notebook 2, upload its test_clean_manifest.csv as a Kaggle '            'dataset, and set CLEAN_MANIFEST to that path (or set it to None '            'to deliberately report the contaminated full-test numbers).')    keep_keys = set(pd.read_csv(_mp)['path'].astype(str).map(_key))    CLEAN_MASK = np.array([k in keep_keys for k in TEST_KEYS])    EVAL_SET = 'clean'    n_keep, n_drop = int(CLEAN_MASK.sum()), int((~CLEAN_MASK).sum())    print(f'\nclean test manifest : {_mp.name}')    print(f'  manifest rows     : {len(keep_keys)}')    print(f'  matched in test   : {n_keep}')    print(f'  removed as dupes  : {n_drop}')    if n_keep == 0:        raise RuntimeError('manifest matched 0 test images -- key format mismatch.')    if n_keep != len(keep_keys):        print(f'  NOTE: {len(keep_keys)-n_keep} manifest rows did not match this '              'test split; check that both notebooks used the same corpus.')# Every downstream section evaluates on these, so the clean subset is primary.def apply_mask(a):    return a[CLEAN_MASK]LABELS['test_full'] = LABELS['test']PATHS['test_full']  = PATHS['test']CACHE['test_full']  = dict(CACHE['test'])LABELS['test'] = LABELS['test'][CLEAN_MASK]PATHS['test']  = PATHS['test'][CLEAN_MASK]CACHE['test']  = {k: v[CLEAN_MASK] for k, v in CACHE['test'].items()}print(f'\nPRIMARY test set: n={len(LABELS["test"])} ({EVAL_SET})')cnt_c = Counter(LABELS['test'].tolist())print('  ' + ' '.join(f'{c}={cnt_c[i]}' for i, c in enumerate(CLASSES)))CLEAN_INFO = dict(eval_set=EVAL_SET, manifest=str(CLEAN_MANIFEST),                  n_full=int(len(CLEAN_MASK)), n_clean=int(CLEAN_MASK.sum()),                  n_removed=int((~CLEAN_MASK).sum()),                  pct_removed=float((~CLEAN_MASK).mean()*100),                  class_counts={CLASSES[i]: int(cnt_c[i]) for i in range(NUM_CLS)})

## 6. Reproduction check — gateAsserts the reproduced single-backbone test accuracy and macro-AUC match the valuesstored in each checkpoint to within **0.01 pp** (1e-4 absolute). If any check failsthe notebook raises, because every downstream number would be suspect.Reference values: EfficientNet 85.11 / 0.9636, ConvNeXt 71.35 / 0.9418,Swin 85.39 / 0.9683.**This gate runs on the FULL 727-image test set**, not the de-duplicated subset:the stored values were measured on all 727, so only that set is comparable. Every*reported* metric downstream uses the clean 565-image subset. Comparing a storedfull-test accuracy against a clean-subset accuracy would fail for a reason that hasnothing to do with reproduction.

In [ ]:
TOL = 1e-4   # 0.01 ppdef macro_auc_of(probs, labels):    yb = label_binarize(labels, classes=list(range(NUM_CLS)))    return roc_auc_score(yb, probs, multi_class='ovr', average='macro')repro_rows, failures = [], []# Gate against the FULL test set -- the basis on which the stored values were computed.y_test_full = LABELS['test_full']y_test = LABELS['test']          # clean subset: used by every section after this oneprint(f'gate basis: full test set n={len(y_test_full)} '      f'(reported metrics use the clean subset n={len(y_test)})')print(f'{"model":16s} {"acc_repro":>10} {"acc_ckpt":>10} {"d_pp":>8} '      f'{"auc_repro":>10} {"auc_ckpt":>10} {"d":>9}')print('-'*80)for key in MODELS:    p   = CACHE['test_full'][key]    acc = accuracy_score(y_test_full, p.argmax(1))    ma  = macro_auc_of(p, y_test_full)    acc_ck = float(CKPTS[key]['test_acc']); auc_ck = float(CKPTS[key]['macro_auc'])    d_acc, d_auc = acc - acc_ck, ma - auc_ck    ok = (abs(d_acc) <= TOL) and (abs(d_auc) <= TOL)    if not ok:        failures.append((MODEL_LONG[key], d_acc, d_auc))    repro_rows.append(dict(model=MODEL_LONG[key], acc_repro=acc, acc_ckpt=acc_ck,                           acc_delta_pp=d_acc*100, auc_repro=ma, auc_ckpt=auc_ck,                           auc_delta=d_auc, within_tol=bool(ok)))    print(f'{MODEL_LONG[key]:16s} {acc*100:>9.4f}% {acc_ck*100:>9.4f}% {d_acc*100:>+8.4f} '          f'{ma:>10.6f} {auc_ck:>10.6f} {d_auc:>+9.6f} {"OK" if ok else "FAIL"}')pd.DataFrame(repro_rows).to_csv(OUT/'reproduction_check.csv', index=False)if failures:    msg = ['REPRODUCTION FAILED - stopping. Mismatches:']    for nm, da, du in failures:        msg.append(f'  {nm}: d_acc={da*100:+.4f} pp, d_auc={du:+.6f}')    msg += [        '', 'Likely causes, in order of probability:',        '  1. DATA_ROOT points at a different corpus or a different test split.',        '  2. Extra/missing files in the test folders (expected n=727 for the 5 classes).',        '  3. timm version changed a backbone default (check timm.__version__).',        '  4. Pillow resampling differences in transforms.Resize.',        '  5. Class order mismatch (checked earlier, so unlikely).',        'Do NOT interpret any downstream metric until this passes.',    ]    raise AssertionError('\n'.join(msg))print('\nReproduction check PASSED for all three backbones.')

## 7. The 7 configurationsEqual-weight mean of the member softmax outputs — exactly what`src/models/ensemble.py` and `ensemble_evaluation.ipynb` do with `WEIGHTS=(1,1,1)`.

In [ ]:
CONFIGS = ['E', 'C', 'S', 'EC', 'ES', 'CS', 'ECS']def config_probs(cfg, split):    # Equal-weight mean of the softmax outputs of the members of `cfg`.    members = [CACHE[split][k] for k in cfg]    return np.mean(members, axis=0)def config_label(cfg):    if len(cfg) == 1:        return MODEL_LONG[cfg]    return ' + '.join(MODEL_LONG[k].split('-')[0] for k in cfg)for cfg in CONFIGS:    print(f'{cfg:4s} -> {config_label(cfg)}')

## 8. Metrics — overall and per class

In [ ]:
def specificity_per_class(cm):    # TN / (TN + FP) for each class, from a confusion matrix.    out = []    total = cm.sum()    for i in range(cm.shape[0]):        fp = cm[:, i].sum() - cm[i, i]        fn = cm[i, :].sum() - cm[i, i]        tp = cm[i, i]        tn = total - tp - fp - fn        out.append(tn/(tn+fp) if (tn+fp) > 0 else float('nan'))    return np.array(out)def all_metrics(probs, labels):    preds = probs.argmax(1)    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLS)))    prec, rec, f1, support = precision_recall_fscore_support(        labels, preds, labels=list(range(NUM_CLS)), zero_division=0)    spec = specificity_per_class(cm)    return dict(        accuracy=float(accuracy_score(labels, preds)),        balanced_accuracy=float(balanced_accuracy_score(labels, preds)),        macro_auc=float(macro_auc_of(probs, labels)),        mcc=float(matthews_corrcoef(labels, preds)),        macro_f1=float(f1_score(labels, preds, average='macro', zero_division=0)),        per_class={            CLASSES[i]: dict(precision=float(prec[i]), recall=float(rec[i]),                             f1=float(f1[i]), specificity=float(spec[i]),                             support=int(support[i]),                             auc=float(roc_auc_score(                                 (labels == i).astype(int), probs[:, i])))            for i in range(NUM_CLS)},        confusion_matrix=cm.tolist(),    )RESULTS = {}print(f'{"config":28s} {"acc":>8} {"bal_acc":>8} {"macroAUC":>9} {"MCC":>8} {"macroF1":>8}')print('-'*76)for cfg in CONFIGS:    m = all_metrics(config_probs(cfg, 'test'), y_test)    RESULTS[cfg] = m    print(f'{cfg+" | "+config_label(cfg):28s} {m["accuracy"]*100:>7.2f}% '          f'{m["balanced_accuracy"]*100:>7.2f}% {m["macro_auc"]:>9.4f} '          f'{m["mcc"]:>8.4f} {m["macro_f1"]:>8.4f}')# flat tablesrows = [dict(config=c, label=config_label(c), **{k: RESULTS[c][k] for k in            ('accuracy','balanced_accuracy','macro_auc','mcc','macro_f1')})        for c in CONFIGS]pd.DataFrame(rows).to_csv(OUT/'ablation_table.csv', index=False)pc_rows = []for c in CONFIGS:    for cls, d in RESULTS[c]['per_class'].items():        pc_rows.append(dict(config=c, label=config_label(c), cls=cls, **d))pd.DataFrame(pc_rows).to_csv(OUT/'per_class_metrics.csv', index=False)print('\nwrote ablation_table.csv, per_class_metrics.csv')print('\nPer-class recall (sensitivity), test set:')hdr = f'{"config":10s}' + ''.join(f'{c[:12]:>14s}' for c in CLASSES)print(hdr); print('-'*len(hdr))for c in CONFIGS:    print(f'{c:10s}' + ''.join(        f'{RESULTS[c]["per_class"][cls]["recall"]:>14.4f}' for cls in CLASSES))

## 9. Bootstrap 95% CIs2000 stratum-free resamples of the test set with replacement, seed 42. The **same**resample index matrix is reused for every configuration, which is what makes thepaired difference in the next cell valid.A resample that loses a class entirely would make macro-AUC undefined; those drawsare skipped for AUC and counted.

In [ ]:
N_BOOT = 2000rng = np.random.default_rng(SEED)n_test = len(y_test)BOOT_IDX = rng.integers(0, n_test, size=(N_BOOT, n_test))print(f'bootstrap: {N_BOOT} resamples x n={n_test}, seed={SEED}')def boot_metrics(probs, labels, idx_matrix):    accs, aucs, mccs = [], [], []    skipped = 0    preds_full = probs.argmax(1)    for idx in idx_matrix:        yb, pb = labels[idx], preds_full[idx]        accs.append((yb == pb).mean())        mccs.append(matthews_corrcoef(yb, pb))        if len(np.unique(yb)) == NUM_CLS:            aucs.append(macro_auc_of(probs[idx], yb))        else:            skipped += 1    return np.array(accs), np.array(aucs), np.array(mccs), skippeddef ci(a):    if len(a) == 0:        return dict(mean=None, lo=None, hi=None, n=0)    return dict(mean=float(np.mean(a)),                lo=float(np.percentile(a, 2.5)),                hi=float(np.percentile(a, 97.5)), n=int(len(a)))BOOT = {}print(f'\n{"config":10s} {"accuracy 95% CI":>30s} {"macro-AUC 95% CI":>30s} {"MCC 95% CI":>30s}')print('-'*104)for cfg in CONFIGS:    p = config_probs(cfg, 'test')    a, u, m, sk = boot_metrics(p, y_test, BOOT_IDX)    BOOT[cfg] = dict(accuracy=ci(a), macro_auc=ci(u), mcc=ci(m), auc_draws_skipped=sk)    print(f'{cfg:10s} '          f'{RESULTS[cfg]["accuracy"]*100:>7.2f}% [{a.mean()*100:5.2f}] ({np.percentile(a,2.5)*100:5.2f}-{np.percentile(a,97.5)*100:5.2f})  '          f'{RESULTS[cfg]["macro_auc"]:>7.4f} ({np.percentile(u,2.5):.4f}-{np.percentile(u,97.5):.4f})  '          f'{RESULTS[cfg]["mcc"]:>7.4f} ({np.percentile(m,2.5):.4f}-{np.percentile(m,97.5):.4f})')    if sk:        print(f'{"":10s}   (AUC: {sk} resample(s) skipped for missing a class)')

### 9b. Paired bootstrap — ensemble {E,C,S} minus SwinThe key question the paper needs answered honestly: is the three-model ensembleactually better than Swin alone? Because both configurations are scored on the*same* resample, this is a paired test. If the CI of the difference includes 0, thepaper must not claim an improvement.

In [ ]:
def paired_diff(cfg_a, cfg_b, idx_matrix):    pa, pb = config_probs(cfg_a, 'test'), config_probs(cfg_b, 'test')    pred_a, pred_b = pa.argmax(1), pb.argmax(1)    d_acc, d_auc, d_mcc = [], [], []    for idx in idx_matrix:        yb = y_test[idx]        d_acc.append((yb == pred_a[idx]).mean() - (yb == pred_b[idx]).mean())        d_mcc.append(matthews_corrcoef(yb, pred_a[idx]) - matthews_corrcoef(yb, pred_b[idx]))        if len(np.unique(yb)) == NUM_CLS:            d_auc.append(macro_auc_of(pa[idx], yb) - macro_auc_of(pb[idx], yb))    return np.array(d_acc), np.array(d_auc), np.array(d_mcc)d_acc, d_auc, d_mcc = paired_diff('ECS', 'S', BOOT_IDX)PAIRED = {}print('Paired bootstrap: {E,C,S} minus Swin-Tiny (same resamples)\n')for nm, arr, obs, scale in [    ('accuracy (pp)', d_acc, RESULTS['ECS']['accuracy']-RESULTS['S']['accuracy'], 100),    ('macro-AUC',     d_auc, RESULTS['ECS']['macro_auc']-RESULTS['S']['macro_auc'], 1),    ('MCC',           d_mcc, RESULTS['ECS']['mcc']-RESULTS['S']['mcc'], 1),]:    lo, hi = np.percentile(arr, 2.5)*scale, np.percentile(arr, 97.5)*scale    frac = float((arr > 0).mean())    crosses = bool(lo <= 0 <= hi)    PAIRED[nm] = dict(observed=float(obs*scale), ci_lo=float(lo), ci_hi=float(hi),                      mean=float(arr.mean()*scale), frac_favouring_ensemble=frac,                      ci_includes_zero=crosses, n=int(len(arr)))    verdict = 'CI INCLUDES 0 -> no significant difference' if crosses else 'CI excludes 0'    print(f'{nm:16s} observed={obs*scale:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  '          f'P(ens>swin)={frac:.3f}   {verdict}')print('\nInterpretation note for the paper: where the CI includes 0, report the')print('difference as an observation, not as an improvement.')

## 10. Confusion matrices — saved as CSV, not only as a plot

In [ ]:
for cfg in CONFIGS:    cm = np.array(RESULTS[cfg]['confusion_matrix'])    cmn = cm / cm.sum(axis=1, keepdims=True)    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT/f'confusion_{cfg}_counts.csv')    pd.DataFrame(cmn, index=CLASSES, columns=CLASSES).to_csv(OUT/f'confusion_{cfg}_rownorm.csv')print('wrote confusion_*_counts.csv and confusion_*_rownorm.csv for all 7 configs')cm = np.array(RESULTS['ECS']['confusion_matrix'])cmn = cm / cm.sum(axis=1, keepdims=True)print('\n{E,C,S} row-normalised confusion matrix (true -> predicted):')print(f'{"true\\pred":26s}' + ''.join(f'{c[:11]:>13s}' for c in CLASSES))for i, c in enumerate(CLASSES):    print(f'{c:26s}' + ''.join(f'{cmn[i,j]:>13.3f}' for j in range(NUM_CLS)))print('\nThe paper claims Eczema->Tinea 22%, Tinea->Eczema 19%, Keratosis->BCC 10%.')for a, b in [('Eczema','Tinea'), ('Tinea','Eczema'), ('Keratosis','Basal_Cell_Carcinoma')]:    v = cmn[CLASSES.index(a), CLASSES.index(b)]    print(f'  measured {a:>10s} -> {b:<22s} {v*100:5.1f}%')

## 11. Agreement analysis`agreement = 1.0` when all three backbones' top-1 equals the ensemble top-1(the definition used in `src/models/ensemble.py`). Reported here as unanimity amongthe three members, plus accuracy on agreed vs. disagreed cases — the quantity thatwould justify routing low-agreement cases to a clinician.

In [ ]:
tops = np.stack([CACHE['test'][k].argmax(1) for k in MODELS], axis=1)   # (n, 3)unanimous = (tops[:, 0] == tops[:, 1]) & (tops[:, 1] == tops[:, 2])ens_pred = config_probs('ECS', 'test').argmax(1)n_agree = int(unanimous.sum()); n_dis = int((~unanimous).sum())acc_agree = float((ens_pred[unanimous] == y_test[unanimous]).mean()) if n_agree else float('nan')acc_dis   = float((ens_pred[~unanimous] == y_test[~unanimous]).mean()) if n_dis else float('nan')# ensemble-referenced agreement tiers (0.33 / 0.67 / 1.0)agree_frac = (tops == ens_pred[:, None]).sum(1) / 3.0tier_rows = []for t in sorted(set(np.round(agree_frac, 2))):    m = np.isclose(agree_frac, t)    tier_rows.append(dict(tier=float(t), n=int(m.sum()),                          pct=float(m.mean()*100),                          accuracy=float((ens_pred[m] == y_test[m]).mean())))AGREEMENT = dict(    n_test=int(n_test),    unanimous_n=n_agree, unanimous_frac=float(unanimous.mean()),    accuracy_when_unanimous=acc_agree,    disagreed_n=n_dis, accuracy_when_disagreed=acc_dis,    tiers=tier_rows,)print(f'all three agree : {n_agree}/{n_test} ({unanimous.mean()*100:.1f}%)  '      f'ensemble acc = {acc_agree*100:.2f}%')print(f'they disagree   : {n_dis}/{n_test} ({(1-unanimous.mean())*100:.1f}%)  '      f'ensemble acc = {acc_dis*100:.2f}%')print('\nagreement tier (fraction matching ensemble top-1):')for r in tier_rows:    print(f'  {r["tier"]:.2f}  n={r["n"]:4d} ({r["pct"]:5.1f}%)  acc={r["accuracy"]*100:6.2f}%')pd.DataFrame(tier_rows).to_csv(OUT/'agreement_tiers.csv', index=False)

## 12. OPTIONAL — weight grid search on VALIDATION only**Secondary result.** Weights are searched on the validation split in steps of 0.1and then applied once to test. The primary reported system remains the equal-weightaverage; this cell only quantifies what tuning could buy. Selecting on test would beleakage, so test is touched exactly once, at the end.

In [ ]:
y_val = LABELS['val']grid = [i/10 for i in range(11)]cands = []for a in grid:    for b in grid:        c = round(1.0 - a - b, 10)        if c < -1e-9:            continue        w = (round(a, 10), round(b, 10), max(c, 0.0))        if abs(sum(w) - 1.0) < 1e-9 and sum(x > 0 for x in w) >= 1:            cands.append(w)pv = {k: CACHE['val'][k] for k in MODELS}pt = {k: CACHE['test'][k] for k in MODELS}best = Nonefor w in cands:    pm = w[0]*pv['E'] + w[1]*pv['C'] + w[2]*pv['S']    a = accuracy_score(y_val, pm.argmax(1))    if best is None or a > best[1]:        best = (w, a)w_best, val_acc_best = bestequal = (1/3, 1/3, 1/3)pm_eq_val  = sum(w*pv[k] for w, k in zip(equal,  MODELS))pm_bs_test = sum(w*pt[k] for w, k in zip(w_best, MODELS))pm_eq_test = sum(w*pt[k] for w, k in zip(equal,  MODELS))WEIGHT_SEARCH = dict(    searched_on='val', n_candidates=len(cands), step=0.1,    best_weights=dict(zip(['E','C','S'], [float(x) for x in w_best])),    val_accuracy_best=float(val_acc_best),    val_accuracy_equal=float(accuracy_score(y_val, pm_eq_val.argmax(1))),    test_accuracy_tuned=float(accuracy_score(y_test, pm_bs_test.argmax(1))),    test_macro_auc_tuned=float(macro_auc_of(pm_bs_test, y_test)),    test_mcc_tuned=float(matthews_corrcoef(y_test, pm_bs_test.argmax(1))),    test_accuracy_equal=float(accuracy_score(y_test, pm_eq_test.argmax(1))),    note='SECONDARY RESULT. Weights chosen on val only; test used once. '         'Primary reported system is the equal-weight average.',)print(f'candidates evaluated : {len(cands)} (step 0.1, weights sum to 1)')print(f'best weights on VAL  : E={w_best[0]:.1f} C={w_best[1]:.1f} S={w_best[2]:.1f}')print(f'  val acc  tuned={WEIGHT_SEARCH["val_accuracy_best"]*100:.2f}%  '      f'equal={WEIGHT_SEARCH["val_accuracy_equal"]*100:.2f}%')print(f'  test acc tuned={WEIGHT_SEARCH["test_accuracy_tuned"]*100:.2f}%  '      f'equal={WEIGHT_SEARCH["test_accuracy_equal"]*100:.2f}%')print(f'  test AUC tuned={WEIGHT_SEARCH["test_macro_auc_tuned"]:.4f}  '      f'MCC tuned={WEIGHT_SEARCH["test_mcc_tuned"]:.4f}')print('\nLabel this clearly as a secondary result in the paper.')

## 12b. Confidence calibration — ECE, MCE and reliability curvesThe triage layer acts on `primary_confidence`: `severity_triage.py` blocks and asksfor a re-capture below 0.40, and escalates a melanoma top-1 to urgent at or above0.40. Those thresholds are only meaningful if the reported confidence behaves like aprobability, so it is measured here rather than assumed.`SkinEnsemble` normalises its weights in `__init__` (`w / sum(weights)`), so theaveraged vector sums to 1 and `primary_confidence` is a genuine probability. What is*not* guaranteed is calibration: deep networks are systematically overconfident, andboth label smoothing (0.1 for EfficientNet, 0.15 for ConvNeXt and Swin) and theclass-weighted loss distort the probability scale further.- **ECE** — Expected Calibration Error, the bin-weighted mean gap between confidence  and accuracy. `< 0.05` is well calibrated, `0.05-0.15` typical for an uncalibrated  deep net, `> 0.15` poor.- **MCE** — the worst single-bin gap.- **Temperature scaling** — one scalar `T` fitted on *validation* by minimising NLL,  applied as `softmax(logit / T)`. It cannot change any argmax, so accuracy, AUC and  MCC are unaffected; it only rescales confidence. Fitted on val, reported on test.Equal-weight averaging happens in probability space, so there is no single ensemblelogit. `T` is therefore fitted on `log(p_ensemble)`, which is the standard treatmentfor an averaged probability vector.

In [ ]:
N_BINS = 10def ece_mce(conf, correct, n_bins=N_BINS):    # Equal-width bins on [0,1]; empty bins contribute nothing.    edges = np.linspace(0.0, 1.0, n_bins + 1)    n = len(conf)    ece = 0.0    mce = 0.0    rows = []    for i in range(n_bins):        lo, hi = edges[i], edges[i + 1]        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)        cnt = int(m.sum())        if cnt == 0:            rows.append(dict(bin_lo=float(lo), bin_hi=float(hi), n=0,                             mean_conf=None, accuracy=None, gap=None))            continue        mc = float(conf[m].mean())        ac = float(correct[m].mean())        gap = abs(ac - mc)        ece += (cnt / n) * gap        mce = max(mce, gap)        rows.append(dict(bin_lo=float(lo), bin_hi=float(hi), n=cnt,                         mean_conf=mc, accuracy=ac, gap=float(ac - mc)))    return float(ece), float(mce), rowsdef fit_temperature(probs_val, y_val, lo=0.25, hi=10.0, iters=60):    # Fit T by golden-section search on validation NLL. Operates on log-probs    # because the ensemble averages probabilities, so no single logit exists.    logp = np.log(np.clip(probs_val, 1e-12, 1.0))    def nll(T):        z = logp / T        z = z - z.max(1, keepdims=True)        p = np.exp(z)        p /= p.sum(1, keepdims=True)        return float(-np.log(np.clip(p[np.arange(len(y_val)), y_val], 1e-12, 1.0)).mean())    gr = (np.sqrt(5.0) - 1.0) / 2.0    a, b = lo, hi    c, d = b - gr * (b - a), a + gr * (b - a)    for _ in range(iters):        if nll(c) < nll(d):            b, d = d, c            c = b - gr * (b - a)        else:            a, c = c, d            d = a + gr * (b - a)    T = (a + b) / 2.0    return float(T), nll(T), nll(1.0)def apply_T(probs, T):    z = np.log(np.clip(probs, 1e-12, 1.0)) / T    z = z - z.max(1, keepdims=True)    p = np.exp(z)    return p / p.sum(1, keepdims=True)CALIB = {}calib_rows = []for c in CONFIGS:    pt_c = config_probs(c, 'test')    conf = pt_c.max(1)    correct = (pt_c.argmax(1) == y_test).astype(float)    ece, mce, bins = ece_mce(conf, correct)    CALIB[c] = dict(label=config_label(c), n_bins=N_BINS,                    ece=ece, mce=mce,                    mean_confidence=float(conf.mean()),                    accuracy=float(correct.mean()),                    overconfidence=float(conf.mean() - correct.mean()),                    bins=bins)    calib_rows.append(dict(config=config_label(c), ece=ece, mce=mce,                           mean_conf=float(conf.mean()),                           accuracy=float(correct.mean()),                           overconfidence=float(conf.mean() - correct.mean())))# Temperature scaling for the headline equal-weight ensemble: fit on val, report on test.pv_ecs = config_probs('ECS', 'val')pt_ecs = config_probs('ECS', 'test')T, nll_T, nll_1 = fit_temperature(pv_ecs, LABELS['val'])pt_T = apply_T(pt_ecs, T)ece_before, mce_before, _ = ece_mce(pt_ecs.max(1), (pt_ecs.argmax(1) == y_test).astype(float))ece_after, mce_after, bins_after = ece_mce(pt_T.max(1), (pt_T.argmax(1) == y_test).astype(float))acc_before = float((pt_ecs.argmax(1) == y_test).mean())acc_after = float((pt_T.argmax(1) == y_test).mean())assert abs(acc_before - acc_after) < 1e-12, (    'temperature scaling must not change any prediction')TEMPERATURE = dict(    T=T, fitted_on='val', applied_to='test',    val_nll_at_T=nll_T, val_nll_at_T1=nll_1,    test_ece_before=ece_before, test_ece_after=ece_after,    test_mce_before=mce_before, test_mce_after=mce_after,    test_accuracy_unchanged=acc_before,    bins_after=bins_after,    note='One scalar fitted on validation NLL. Cannot change argmax, so accuracy, '         'macro-AUC and MCC are identical before and after; only confidence is rescaled.',)pd.DataFrame(calib_rows).to_csv(OUT/'calibration_table.csv', index=False)pd.DataFrame(CALIB['ECS']['bins']).to_csv(OUT/'reliability_bins_ecs.csv', index=False)print(f'{"config":22s} {"ECE":>7} {"MCE":>7} {"mean conf":>10} {"acc":>7} {"overconf":>9}')print('-' * 68)for r in calib_rows:    print(f'{r["config"]:22s} {r["ece"]:7.4f} {r["mce"]:7.4f} '          f'{r["mean_conf"]*100:9.2f}% {r["accuracy"]*100:6.2f}% '          f'{r["overconfidence"]*100:+8.2f}pp')print(f'\n--- temperature scaling (equal-weight ensemble) ---')print(f'  T fitted on val      : {T:.4f}   (T>1 means the model was overconfident)')print(f'  val NLL  T=1 -> T    : {nll_1:.4f} -> {nll_T:.4f}')print(f'  test ECE before/after: {ece_before:.4f} -> {ece_after:.4f}')print(f'  test MCE before/after: {mce_before:.4f} -> {mce_after:.4f}')print(f'  test accuracy        : {acc_before*100:.4f}% (unchanged, by construction)')print('\n--- reliability, equal-weight ensemble (test) ---')print(f'{"bin":>14} {"n":>5} {"mean conf":>10} {"accuracy":>9} {"gap":>8}')for b in CALIB['ECS']['bins']:    if b['n'] == 0:        continue    print(f'  ({b["bin_lo"]:.1f}, {b["bin_hi"]:.1f}] {b["n"]:5d} '          f'{b["mean_conf"]*100:9.2f}% {b["accuracy"]*100:8.2f}% {b["gap"]*100:+7.2f}pp')# How many test images would the triage thresholds actually touch?thr = 0.40below = int((pt_ecs.max(1) < thr).sum())mel_idx = CLASSES.index('Melanoma')mel_top = pt_ecs.argmax(1) == mel_idxmel_esc = int((mel_top & (pt_ecs.max(1) >= thr)).sum())TRIAGE_REACH = dict(    threshold=thr,    n_below_threshold=below,    pct_below_threshold=float(below / len(y_test) * 100),    n_melanoma_top1=int(mel_top.sum()),    n_melanoma_escalated=mel_esc,    note='Reachability of the severity_triage.py thresholds on the real test set, '         'using the equal-weight ensemble confidence.',)print(f'\n--- triage threshold reach (test, equal-weight) ---')print(f'  confidence < {thr:.2f} (re-capture) : {below} / {len(y_test)} '      f'({below/len(y_test)*100:.1f}%)')print(f'  melanoma top-1 at >= {thr:.2f}      : {mel_esc} / {int(mel_top.sum())}')

## 13. Figures — PDF (every value also lives in a CSV/JSON)

In [ ]:
sns.set_style('whitegrid')MAIN4 = ['E', 'C', 'S', 'ECS']labels4 = ['EfficientNet-B1', 'ConvNeXt-Tiny', 'Swin-Tiny', 'Ensemble (E+C+S)']bar_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']# ---- fig3: model comparison with bootstrap CI error bars ----fig, axes = plt.subplots(1, 3, figsize=(15, 5))for ax, metric, ylab, title, fmt, scale in zip(        axes, ['accuracy', 'macro_auc', 'mcc'],        ['Accuracy (%)', 'Macro-average AUC', 'MCC'],        ['Test Accuracy', 'Macro-Average AUC', 'Matthews Correlation Coefficient'],        ['{:.2f}%', '{:.4f}', '{:.4f}'], [100, 1, 1]):    vals = [RESULTS[c][metric]*scale for c in MAIN4]    los  = [(RESULTS[c][metric]-BOOT[c][metric]['lo'])*scale for c in MAIN4]    his  = [(BOOT[c][metric]['hi']-RESULTS[c][metric])*scale for c in MAIN4]    x = np.arange(len(MAIN4))    bars = ax.bar(x, vals, color=bar_colors, edgecolor='black', linewidth=0.7, width=0.55,                  yerr=[los, his], capsize=4, error_kw=dict(lw=1, ecolor='black'))    for b, v in zip(bars, vals):        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.015,                fmt.format(v), ha='center', va='bottom', fontsize=8.5, fontweight='bold')    ax.set_xticks(x); ax.set_xticklabels(labels4, rotation=18, ha='right', fontsize=8.5)    ax.set_ylabel(ylab, fontsize=10); ax.set_title(title, fontweight='bold', fontsize=11)    ax.set_ylim(0, max(np.array(vals)+np.array(his))*1.15)    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)plt.suptitle('Model comparison with bootstrap 95% CIs (equal-weight softmax averaging)',             fontsize=12.5, fontweight='bold')plt.tight_layout()plt.savefig(OUT/'fig3_model_comparison.pdf', bbox_inches='tight')plt.close()# ---- fig4: ROC, per class + macro ----fig, axes = plt.subplots(2, 3, figsize=(17, 10.5)); axes = axes.flatten()series = [(MODEL_LONG['E'], config_probs('E','test'), '--'),          (MODEL_LONG['C'], config_probs('C','test'), '-.'),          (MODEL_LONG['S'], config_probs('S','test'), ':'),          ('Ensemble (E+C+S)', config_probs('ECS','test'), '-')]for i, cls in enumerate(CLASSES):    ax = axes[i]    for nm, probs, ls in series:        fpr, tpr, _ = roc_curve((y_test == i).astype(int), probs[:, i])        ax.plot(fpr, tpr, linestyle=ls, lw=2.3 if ls == '-' else 1.4,                label=f'{nm} (AUC={auc(fpr,tpr):.3f})')    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)    ax.set_title(cls.replace('_',' '), fontweight='bold', fontsize=10.5)    ax.set_xlabel('False positive rate', fontsize=9); ax.set_ylabel('True positive rate', fontsize=9)    ax.legend(fontsize=7.5, loc='lower right'); ax.grid(alpha=0.3)ax = axes[5]mean_fpr = np.linspace(0, 1, 300)for nm, probs, ls in series:    tprs = [np.interp(mean_fpr, *roc_curve((y_test == j).astype(int), probs[:, j])[:2])            for j in range(NUM_CLS)]    ax.plot(mean_fpr, np.mean(tprs, axis=0), linestyle=ls, lw=2.3 if ls == '-' else 1.4,            label=f'{nm} (macro={macro_auc_of(probs, y_test):.3f})')ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)ax.set_title('Macro-average', fontweight='bold', fontsize=10.5)ax.set_xlabel('False positive rate', fontsize=9); ax.set_ylabel('True positive rate', fontsize=9)ax.legend(fontsize=7.5, loc='lower right'); ax.grid(alpha=0.3)plt.suptitle('ROC curves — individual backbones vs. equal-weight ensemble', fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig(OUT/'fig4_roc.pdf', bbox_inches='tight')plt.close()# ---- fig5: confusion matrices ----fig, axes = plt.subplots(1, 2, figsize=(16, 6))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES,            ax=axes[0], linewidths=0.5, linecolor='gray', cbar_kws={'shrink':0.8})axes[0].set_title('Counts', fontsize=12, fontweight='bold')sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES,            ax=axes[1], vmin=0, vmax=1, linewidths=0.5, linecolor='gray', cbar_kws={'shrink':0.8})axes[1].set_title('Row-normalised', fontsize=12, fontweight='bold')for ax in axes:    ax.set_xlabel('Predicted', fontsize=10); ax.set_ylabel('True', fontsize=10)    ax.tick_params(axis='x', rotation=30); ax.tick_params(axis='y', rotation=0)plt.suptitle(f'Ensemble (E+C+S) confusion matrix — test n={n_test}, '             f'accuracy {RESULTS["ECS"]["accuracy"]*100:.2f}%', fontsize=12.5, fontweight='bold')plt.tight_layout()plt.savefig(OUT/'fig5_confusion_matrix.pdf', bbox_inches='tight')plt.close()# ---- fig7: reliability diagram + confidence histogram ----fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))ax = axes[0]ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6, label='Perfect calibration')for c, ls, mk in [('E', '--', 'o'), ('C', '-.', 's'), ('S', ':', '^'), ('ECS', '-', 'D')]:    bs = [b for b in CALIB[c]['bins'] if b['n'] > 0]    ax.plot([b['mean_conf'] for b in bs], [b['accuracy'] for b in bs],            linestyle=ls, marker=mk, ms=5, lw=2.2 if c == 'ECS' else 1.3,            label=f'{config_label(c)} (ECE={CALIB[c]["ece"]:.3f})')bs_T = [b for b in TEMPERATURE['bins_after'] if b['n'] > 0]ax.plot([b['mean_conf'] for b in bs_T], [b['accuracy'] for b in bs_T],        linestyle='-', marker='*', ms=8, lw=2.2, color='crimson',        label=f'Ensemble, T={TEMPERATURE["T"]:.2f} (ECE={TEMPERATURE["test_ece_after"]:.3f})')ax.set_xlabel('Mean predicted confidence', fontsize=10)ax.set_ylabel('Observed accuracy', fontsize=10)ax.set_title('Reliability diagram (test)', fontsize=12, fontweight='bold')ax.legend(fontsize=7.5, loc='upper left'); ax.grid(alpha=0.3)ax.set_xlim(0, 1); ax.set_ylim(0, 1)ax = axes[1]conf_ecs = config_probs('ECS', 'test').max(1)ax.hist(conf_ecs, bins=np.linspace(0, 1, N_BINS + 1), edgecolor='black',        alpha=0.8, label='Equal-weight ensemble')ax.axvline(0.40, color='crimson', ls='--', lw=1.8,           label='triage threshold 0.40')ax.set_xlabel('Primary confidence', fontsize=10)ax.set_ylabel('Test images', fontsize=10)ax.set_title(f'Confidence distribution — {TRIAGE_REACH["n_below_threshold"]} of '             f'{n_test} below 0.40', fontsize=12, fontweight='bold')ax.legend(fontsize=8.5); ax.grid(alpha=0.3, axis='y')plt.suptitle('Confidence calibration of the equal-weight ensemble',             fontsize=12.5, fontweight='bold')plt.tight_layout()plt.savefig(OUT/'fig7_calibration.pdf', bbox_inches='tight')plt.close()for f in ['fig3_model_comparison.pdf','fig4_roc.pdf','fig5_confusion_matrix.pdf',          'fig7_calibration.pdf']:    print(f'wrote {f}  ({(OUT/f).stat().st_size//1024} KB)')

## 14. results.json

In [ ]:
payload = dict(    meta=dict(        notebook='01_eval_and_ablation.ipynb',        generated_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),        seed=SEED, gpu=GPU_NAME,        torch=torch.__version__, timm=timm.__version__,        cudnn_deterministic=bool(torch.backends.cudnn.deterministic),        data_root=str(DATA_ROOT), ckpt_root=str(CKPT_ROOT),        classes=CLASSES,        architecture_note=(            'Three independently fine-tuned backbones combined by an EQUAL-WEIGHT '            'mean of their softmax outputs. No fusion network, no learned '            'projections, no fusion MLP exists in any checkpoint.'),        eval_transform='Resize((224,224)) -> ToTensor -> Normalize(ImageNet)',        test_set_note=(            'Every metric in this file is computed on the DE-DUPLICATED test set. '            'Notebook 2 flagged test images with a perceptual-hash near-duplicate '            '(pHash, Hamming d<=5) in train or val and those were removed. The '            'reproduction gate in section 6 is the one exception: it runs on the '            'full test set because the checkpoints stored their accuracy on that '            'basis. Full-test numbers are in results_full_contaminated.json for '            'reference only.'),    ),    deduplication=CLEAN_INFO,    splits=dict(n=SPLIT_N, class_counts=SPLIT_COUNTS),    checkpoint_reference={        MODEL_LONG[k]: dict(            timm_model_name=CKPTS[k]['config']['model_name'],            test_acc=float(CKPTS[k]['test_acc']),            macro_auc=float(CKPTS[k]['macro_auc']),            best_val_acc=float(CKPTS[k]['best_val_acc']),            best_melanoma_sens_val=float(CKPTS[k]['best_melanoma_sens']),            best_composite_val=float(CKPTS[k]['best_composite']),            epochs_actually_run=len(CKPTS[k]['history']['train_loss']),            epochs_configured=CKPTS[k]['config']['num_epochs'],        ) for k in MODELS},    reproduction_check=repro_rows,    configs={c: dict(label=config_label(c), members=list(c),                     metrics=RESULTS[c], bootstrap_ci=BOOT[c]) for c in CONFIGS},    paired_bootstrap_ensemble_minus_swin=PAIRED,    agreement=AGREEMENT,    weight_search_secondary=WEIGHT_SEARCH,    calibration=CALIB,    temperature_scaling=TEMPERATURE,    triage_threshold_reach=TRIAGE_REACH,    bootstrap=dict(n_resamples=N_BOOT, seed=SEED, method='percentile',                   paired='same resample indices across configs'),)with open(OUT/'results.json', 'w', encoding='utf-8') as f:    json.dump(payload, f, indent=2)print(f'wrote results.json ({(OUT/"results.json").stat().st_size//1024} KB)')print('\n--- headline, for the paper ---')for c in ['E','C','S','ECS']:    m, b = RESULTS[c], BOOT[c]    print(f'{config_label(c):22s} acc={m["accuracy"]*100:5.2f}% '          f'[{b["accuracy"]["lo"]*100:.2f}, {b["accuracy"]["hi"]*100:.2f}]  '          f'AUC={m["macro_auc"]:.4f} [{b["macro_auc"]["lo"]:.4f}, {b["macro_auc"]["hi"]:.4f}]  '          f'MCC={m["mcc"]:.4f} [{b["mcc"]["lo"]:.4f}, {b["mcc"]["hi"]:.4f}]')print(f'\ntest n = {n_test}; all metrics on the held-out test split.')

## 15. Contamination appendix — the full-test numbers, for reference onlySections 7–14 above report the **clean 565-image** test set, which is the paper'sprimary result. This cell recomputes the same metrics on the **full 727-image** set— the contaminated one — purely to quantify how much the leakage inflated things.Do **not** report these as results. The single useful number here is the gap: it isthe measurement bias that the pHash screening removed, and one sentence quoting it("de-duplication lowered ensemble accuracy by X pp") is what justifies the screeningin Sec. 4. If `CLEAN_MANIFEST` is None there is no clean subset and this cellno-ops, because the numbers above would already be the full-test ones.

In [ ]:
if EVAL_SET == 'full':    print('EVAL_SET == "full" -> sections 7-14 already report the full test set.')    print('No clean subset to compare against; nothing to do here.')else:    rng_f = np.random.default_rng(SEED)    n_full = len(LABELS['test_full'])    idx_f = rng_f.integers(0, n_full, size=(N_BOOT, n_full))    y_full = LABELS['test_full']    def full_probs(cfg):        # Same equal-weight averaging as config_probs, on the unmasked cache.        return sum(CACHE['test_full'][k] for k in cfg) / len(cfg)    full_results, full_boot = {}, {}    print(f'{"config":10s} {"acc_full":>9} {"acc_clean":>10} {"infl_pp":>9} '          f'{"AUC_full":>9} {"AUC_clean":>10}')    print('-'*62)    for cfg in CONFIGS:        pf = full_probs(cfg)        m = all_metrics(pf, y_full)        a, u, mm, sk = boot_metrics(pf, y_full, idx_f)        full_results[cfg] = m        full_boot[cfg] = dict(accuracy=ci(a), macro_auc=ci(u), mcc=ci(mm),                              auc_draws_skipped=sk)        infl = (m['accuracy'] - RESULTS[cfg]['accuracy']) * 100        print(f'{cfg:10s} {m["accuracy"]*100:>8.2f}% {RESULTS[cfg]["accuracy"]*100:>9.2f}% '              f'{infl:>+9.2f} {m["macro_auc"]:>9.4f} {RESULTS[cfg]["macro_auc"]:>10.4f}')    ens_infl = (full_results['ECS']['accuracy'] - RESULTS['ECS']['accuracy']) * 100    print(f'\nleakage inflation, equal-weight ensemble: {ens_infl:+.2f} pp '          f'({full_results["ECS"]["accuracy"]*100:.2f}% on 727 '          f'-> {RESULTS["ECS"]["accuracy"]*100:.2f}% on 565)')    print('Quote this gap in Sec. 4 to justify the de-duplication; '          'report only the clean numbers as results.')    with open(OUT/'results_full_contaminated.json', 'w', encoding='utf-8') as f:        json.dump(dict(            meta=dict(note='FULL test set -- CONTAMINATED, reference only. '                           'Primary results are on the clean subset '                           '(see results.json).',                      n_full=n_full, n_clean=int(CLEAN_MASK.sum()),                      n_removed=int((~CLEAN_MASK).sum()),                      pct_removed=float((~CLEAN_MASK).mean()*100),                      seed=SEED, n_resamples=N_BOOT),            inflation_pp={c: float((full_results[c]['accuracy']                                    - RESULTS[c]['accuracy'])*100) for c in CONFIGS},            configs={c: dict(label=config_label(c), metrics=full_results[c],                             bootstrap_ci=full_boot[c]) for c in CONFIGS},        ), f, indent=2)    pd.DataFrame([dict(config=c,                       accuracy_full=full_results[c]['accuracy'],                       accuracy_clean=RESULTS[c]['accuracy'],                       inflation_pp=(full_results[c]['accuracy']                                     - RESULTS[c]['accuracy'])*100,                       macro_auc_full=full_results[c]['macro_auc'],                       macro_auc_clean=RESULTS[c]['macro_auc'])                  for c in CONFIGS]).to_csv(OUT/'contamination_comparison.csv', index=False)    print('\nwrote results_full_contaminated.json, contamination_comparison.csv')